# 01. 데이터 수집
서울 6개구(노원구/은평구/서대문구/서초구/강남구/송파구) 아파트 매매가격지수 예측을 위한 데이터 수집

## 수집 데이터
1. **한국부동산원 아파트 매매가격지수** (구별 주간)
2. **한국은행 기준금리 / 주택담보대출 금리** (ECOS API)
3. **KOSPI 지수** (yfinance)
4. **원달러 환율** (yfinance)
5. **전세가격지수** (한국부동산원)

> **사전 준비**: ECOS API 키를 `.env` 파일에 `ECOS_API_KEY=발급받은키` 형식으로 저장하세요.
> 발급: https://ecos.bok.or.kr → 회원가입 → 개발자 → API 서비스 신청 (무료)

In [ ]:
import pandas as pd
import numpy as np
import requests
import yfinance as yf
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

DATA_RAW = Path('../data/raw')
DATA_RAW.mkdir(parents=True, exist_ok=True)

ECOS_API_KEY = os.getenv('ECOS_API_KEY', 'YOUR_ECOS_API_KEY')
TARGET_DISTRICTS = ['노원구', '은평구', '서대문구', '서초구', '강남구', '송파구']
START_DATE = '2015-01-01'
END_DATE   = '2026-05-13'
PREDICT_DATE = '2026-05-25'

print('데이터 저장 경로:', DATA_RAW.resolve())

## 1. 한국부동산원 아파트 매매가격지수

**수동 다운로드 방법** (API 없이 빠른 방법):
1. https://www.reb.or.kr/r-one/main.do 접속
2. 통계 → 주택가격동향조사 → 아파트 매매가격지수
3. 지역: 서울 6개구 선택, 주간 데이터 다운로드
4. `data/raw/reb_apt_price_index.xlsx` 로 저장

**또는 공공데이터포털 API**:

In [ ]:
def load_reb_data_from_excel(file_path='../data/raw/reb_apt_price_index.xlsx'):
    """한국부동산원에서 수동 다운로드한 엑셀 파일 로드.
    
    엑셀 컬럼 구조 예시:
      날짜 | 노원구 | 은평구 | 서대문구 | 서초구 | 강남구 | 송파구
    """
    try:
        df = pd.read_excel(file_path, index_col=0, parse_dates=True)
        df.index.name = 'date'
        # 구 이름 정규화
        df.columns = [c.strip() for c in df.columns]
        available = [c for c in TARGET_DISTRICTS if c in df.columns]
        df = df[available]
        print(f'부동산원 데이터 로드 완료: {df.shape}, 기간: {df.index.min()} ~ {df.index.max()}')
        return df
    except FileNotFoundError:
        print('엑셀 파일이 없습니다. 아래 샘플 데이터를 사용합니다 (실제 과제 시 교체 필요).')
        return generate_sample_price_index()


def generate_sample_price_index():
    """실제 데이터 없을 때 구조 확인용 샘플 생성. 실제 과제에서는 사용 금지."""
    dates = pd.date_range('2020-01-01', '2026-05-13', freq='W-MON')
    np.random.seed(42)

    # 실제 경향 반영: 강남 > 송파 > 서초 > 서대문 > 은평 > 노원
    base = {'노원구': 95, '은평구': 97, '서대문구': 100, '서초구': 110, '강남구': 115, '송파구': 112}
    trend = {'노원구': 0.01, '은평구': 0.01, '서대문구': 0.012, '서초구': 0.015, '강남구': 0.018, '송파구': 0.016}

    records = {}
    for district in TARGET_DISTRICTS:
        t = np.arange(len(dates))
        noise = np.random.normal(0, 0.3, len(dates)).cumsum()
        records[district] = base[district] + trend[district] * t + noise

    df = pd.DataFrame(records, index=dates)
    df.index.name = 'date'
    print(f'샘플 데이터 생성: {df.shape}')
    return df


price_df = load_reb_data_from_excel()
price_df.tail()

## 2. KOSPI 지수 & 원달러 환율 (yfinance)

In [ ]:
def fetch_market_data(start=START_DATE, end=END_DATE):
    tickers = {'^KS11': 'kospi', 'KRW=X': 'usd_krw'}
    result = {}

    for ticker, name in tickers.items():
        data = yf.download(ticker, start=start, end=end, progress=False)
        result[name] = data['Close'].rename(name)
        print(f'{name}: {len(data)}행 수집 ({data.index.min().date()} ~ {data.index.max().date()})')

    market_df = pd.concat(result.values(), axis=1)
    market_df.index.name = 'date'
    return market_df


market_df = fetch_market_data()
market_df.tail()

## 3. 한국은행 기준금리 & 주택담보대출 금리 (ECOS API)

ECOS 통계 코드:
- 기준금리: `722Y001` / 항목 `0101000`
- 주택담보대출 가중평균금리: `028Y003` / 항목 `BECBLA14`

In [ ]:
def fetch_ecos(stat_code, item_code, start_ym, end_ym, api_key=ECOS_API_KEY, freq='MM'):
    """한국은행 ECOS API에서 월별 데이터 수집."""
    url = (
        f'https://ecos.bok.or.kr/api/StatisticSearch/{api_key}/json/kr'
        f'/1/1000/{stat_code}/{freq}/{start_ym}/{end_ym}/{item_code}'
    )
    resp = requests.get(url, timeout=10)
    data = resp.json()

    if 'StatisticSearch' not in data:
        print(f'ECOS API 오류 ({stat_code}): {data}')
        return pd.Series(dtype=float)

    rows = data['StatisticSearch']['row']
    series = pd.Series(
        {r['TIME']: float(r['DATA_VALUE']) for r in rows}
    )
    series.index = pd.to_datetime(series.index, format='%Y%m')
    return series


if ECOS_API_KEY == 'YOUR_ECOS_API_KEY':
    print('.env 파일에 ECOS_API_KEY를 설정하면 실제 금리 데이터를 수집합니다.')
    print('임시 샘플 금리 데이터 생성...')
    dates_m = pd.date_range('2015-01', '2026-05', freq='MS')
    # BOK 기준금리 실제 흐름 근사
    base_rate = pd.Series(
        [1.5]*24 + [1.25]*12 + [0.5]*18 + [3.5]*18 + [3.0]*12 + [2.75]*6 + [2.5]*6,
        index=dates_m[:len(dates_m)]
    )[:len(dates_m)]
    mortgage_rate = base_rate + 1.8 + np.random.normal(0, 0.1, len(base_rate))
    rate_df = pd.DataFrame({'base_rate': base_rate, 'mortgage_rate': mortgage_rate})
else:
    start_ym = START_DATE[:4] + START_DATE[5:7]  # '201501'
    end_ym   = '202605'
    base_rate    = fetch_ecos('722Y001', '0101000',  start_ym, end_ym)
    mortgage_rate = fetch_ecos('028Y003', 'BECBLA14', start_ym, end_ym)
    rate_df = pd.DataFrame({'base_rate': base_rate, 'mortgage_rate': mortgage_rate})

rate_df.index.name = 'date'
print(f'금리 데이터: {rate_df.shape}')
rate_df.tail()

## 4. 전세가격지수 (한국부동산원)

매매/전세 괴리율이 미래 매매가에 선행하는 지표로 사용됩니다.
수동 다운로드: 부동산원 R-ONE → 주택가격동향 → 전세가격지수

In [ ]:
def load_jeonse_data(file_path='../data/raw/reb_jeonse_index.xlsx'):
    try:
        df = pd.read_excel(file_path, index_col=0, parse_dates=True)
        df.columns = [c.strip() + '_전세' for c in df.columns]
        print(f'전세 데이터 로드 완료: {df.shape}')
        return df
    except FileNotFoundError:
        print('전세 엑셀 파일 없음 → 매매가지수 * 0.65 추정값 사용 (과제용 임시)')
        jeonse_df = (price_df * 0.65).copy()
        jeonse_df.columns = [c + '_전세' for c in jeonse_df.columns]
        return jeonse_df


jeonse_df = load_jeonse_data()
jeonse_df.tail()

## 5. 데이터 병합 및 저장

In [ ]:
# 주간 기준으로 통일 (월간 데이터는 ffill로 보간)
weekly_index = pd.date_range(START_DATE, END_DATE, freq='W-MON')

price_weekly  = price_df.reindex(weekly_index).interpolate('linear')
jeonse_weekly = jeonse_df.reindex(weekly_index).interpolate('linear')
market_weekly = market_df.resample('W-MON').last().reindex(weekly_index).ffill()
rate_weekly   = rate_df.resample('W-MON').last().reindex(weekly_index).ffill()

combined = pd.concat([price_weekly, jeonse_weekly, market_weekly, rate_weekly], axis=1)
combined.index.name = 'date'
combined.dropna(how='all', inplace=True)

combined.to_csv(DATA_RAW / 'combined_weekly.csv')
print(f'병합 완료: {combined.shape}')
print(combined.columns.tolist())
combined.tail()

In [ ]:
# 결측치 현황
missing = combined.isnull().sum()
print('결측치 수:\n', missing[missing > 0])
print(f'\n총 행수: {len(combined)}, 기간: {combined.index.min().date()} ~ {combined.index.max().date()}')